In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
REPORT_DIR = PROJECT_DIR / "reports" / "inventory"

BASELINE_PATH = (
    REPORT_DIR / "product_demand_reorder_baseline_original_stockcodes.csv"
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not BASELINE_PATH.exists():
    raise FileNotFoundError(
        f"Original-StockCode baseline not found: {BASELINE_PATH}"
    )

inventory = pd.read_csv(
    BASELINE_PATH,
    dtype={"StockCode": "string"}
)

required_columns = [
    "StockCode",
    "Description",
    "Average_Daily_Units",
    "Daily_Units_Std",
    "Days_With_Sales",
    "Forecast_30Day_Revenue",
    "Forecast_30Day_Demand_Units",
    "Review_Status",
    "Invoice_581483_Units"
]

missing = [
    col for col in required_columns
    if col not in inventory.columns
]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# --------------------------------------------------
# 1. ABC classification by estimated 30-day revenue
# --------------------------------------------------
# This revenue is based on the historical average in the
# 90-day window, not on the product-level forecasting models.
inventory["Forecast_30Day_Revenue"] = pd.to_numeric(
    inventory["Forecast_30Day_Revenue"], errors="coerce"
).fillna(0).clip(lower=0)

inventory = inventory.sort_values(
    "Forecast_30Day_Revenue",
    ascending=False
).reset_index(drop=True)

total_revenue_proxy = inventory["Forecast_30Day_Revenue"].sum()

if total_revenue_proxy <= 0:
    raise ValueError("Estimated product revenue is zero; cannot calculate ABC classes.")

inventory["Cumulative_Revenue_Share_Percent"] = (
    inventory["Forecast_30Day_Revenue"].cumsum()
    / total_revenue_proxy * 100
)

inventory["ABC_Class"] = np.select(
    [
        inventory["Cumulative_Revenue_Share_Percent"] <= 80,
        inventory["Cumulative_Revenue_Share_Percent"] <= 95
    ],
    ["A", "B"],
    default="C"
)

# --------------------------------------------------
# 2. XYZ classification by day-to-day demand variability
# --------------------------------------------------
inventory["Average_Daily_Units"] = pd.to_numeric(
    inventory["Average_Daily_Units"], errors="coerce"
)
inventory["Daily_Units_Std"] = pd.to_numeric(
    inventory["Daily_Units_Std"], errors="coerce"
).fillna(0)

inventory["Days_With_Sales"] = pd.to_numeric(
    inventory["Days_With_Sales"], errors="coerce"
).fillna(0)

inventory["Demand_Coefficient_of_Variation"] = (
    inventory["Daily_Units_Std"]
    / inventory["Average_Daily_Units"].replace(0, np.nan)
)

# Thresholds are practical project assumptions:
# X = relatively stable, Y = moderate variation, Z = high variation.
# I = intermittent/very sparse sales within the 90-day window.
inventory["XYZ_Class"] = np.select(
    [
        (inventory["Average_Daily_Units"] <= 0)
        | (inventory["Days_With_Sales"] < 5),

        inventory["Demand_Coefficient_of_Variation"] <= 0.5,

        inventory["Demand_Coefficient_of_Variation"] <= 1.0
    ],
    ["I", "X", "Y"],
    default="Z"
)

inventory["ABC_XYZ_Class"] = (
    inventory["ABC_Class"] + "-" + inventory["XYZ_Class"]
)

# --------------------------------------------------
# 3. Proposed review priorities
# --------------------------------------------------
manual_review = (
    inventory["Invoice_581483_Units"].fillna(0).gt(0)
    | inventory["Review_Status"].fillna("").str.contains(
        "MANUAL REVIEW", case=False, regex=False
    )
)

other_review = inventory["Review_Status"].fillna("").str.contains(
    "REVIEW", case=False, regex=False
)

high_priority = (
    inventory["ABC_Class"].eq("A")
    & inventory["XYZ_Class"].isin(["Y", "Z", "I"])
)

medium_priority = (
    inventory["ABC_Class"].eq("B")
    & inventory["XYZ_Class"].isin(["Z", "I"])
)

inventory["Proposed_Review_Priority"] = np.select(
    [
        manual_review,
        other_review,
        high_priority,
        medium_priority,
        inventory["ABC_Class"].eq("A")
    ],
    [
        "Manual investigation",
        "Review demand variability",
        "High",
        "Medium-High",
        "High"
    ],
    default="Standard"
)

inventory["Current_Stock_Available"] = False
inventory["Actual_Lead_Time_Available"] = False
inventory["Reorder_Quantity_Calculable"] = False

# --------------------------------------------------
# 4. Save classification and summaries
# --------------------------------------------------
classification_path = REPORT_DIR / "inventory_abc_xyz_analysis.csv"
priority_path = REPORT_DIR / "inventory_review_priority.csv"
summary_path = REPORT_DIR / "inventory_abc_xyz_summary.csv"

inventory.to_csv(classification_path, index=False)

inventory.sort_values(
    ["Proposed_Review_Priority", "Forecast_30Day_Revenue"],
    ascending=[True, False]
).to_csv(priority_path, index=False)

abc_summary = (
    inventory.groupby("ABC_Class", as_index=False)
    .agg(
        Products=("StockCode", "nunique"),
        Estimated_30Day_Revenue=("Forecast_30Day_Revenue", "sum")
    )
)

abc_summary["Product_Share_Percent"] = (
    abc_summary["Products"] / len(inventory) * 100
)
abc_summary["Revenue_Share_Percent"] = (
    abc_summary["Estimated_30Day_Revenue"]
    / total_revenue_proxy * 100
)

xyz_summary = (
    inventory.groupby("XYZ_Class", as_index=False)
    .agg(
        Products=("StockCode", "nunique"),
        Estimated_30Day_Revenue=("Forecast_30Day_Revenue", "sum")
    )
)
xyz_summary["Product_Share_Percent"] = (
    xyz_summary["Products"] / len(inventory) * 100
)

class_matrix = pd.crosstab(
    inventory["ABC_Class"],
    inventory["XYZ_Class"]
).reindex(index=["A", "B", "C"], columns=["X", "Y", "Z", "I"], fill_value=0)

priority_summary = (
    inventory["Proposed_Review_Priority"]
    .value_counts()
    .rename_axis("Proposed_Review_Priority")
    .reset_index(name="Products")
)

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("RETAILPULSE - ABC SUMMARY\n")
    f.write(abc_summary.to_string(index=False))
    f.write("\n\nRETAILPULSE - XYZ SUMMARY\n")
    f.write(xyz_summary.to_string(index=False))
    f.write("\n\nRETAILPULSE - ABC/XYZ MATRIX\n")
    f.write(class_matrix.to_string())
    f.write("\n\nRETAILPULSE - REVIEW PRIORITY SUMMARY\n")
    f.write(priority_summary.to_string(index=False))

print("=" * 70)
print("RETAILPULSE - ABC/XYZ INVENTORY ANALYSIS")
print("=" * 70)

print(f"Products classified: {inventory['StockCode'].nunique():,}")
print(f"Revenue proxy total: {total_revenue_proxy:,.2f}")
print("\nABC summary:")
display(abc_summary.round(2))

print("\nXYZ summary:")
display(xyz_summary.round(2))

print("\nABC/XYZ matrix (product counts):")
display(class_matrix)

print("\nReview priority summary:")
display(priority_summary)

print("\nTop 15 products needing attention:")
display(
    inventory.sort_values(
        ["Proposed_Review_Priority", "Forecast_30Day_Revenue"],
        ascending=[True, False]
    )[[
        "StockCode", "Description", "ABC_XYZ_Class",
        "Forecast_30Day_Revenue",
        "Demand_Coefficient_of_Variation",
        "Days_With_Sales", "Proposed_Review_Priority",
        "Review_Status"
    ]].head(15).round(2)
)

print("\nSaved files:")
print(classification_path)
print(priority_path)
print(summary_path)

print("\nThe master dataset was not modified.")
print(
    "Note: actual reorder quantities still require current stock-on-hand "
    "and actual supplier lead times."
)


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
REPORT_DIR = PROJECT_DIR / "reports" / "inventory"

INPUT_PATH = REPORT_DIR / "inventory_abc_xyz_analysis.csv"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"ABC-XYZ analysis not found: {INPUT_PATH}")

inventory = pd.read_csv(
    INPUT_PATH,
    dtype={"StockCode": "string"}
)

required = [
    "StockCode", "Description", "ABC_Class", "XYZ_Class",
    "Forecast_30Day_Revenue", "Forecast_30Day_Demand_Units",
    "Current_Stock_On_Hand", "Actual_Supplier_Lead_Time_Days",
    "Invoice_581483_Units"
]
missing = [c for c in required if c not in inventory.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# --------------------------------------------------
# 1. Define an inventory policy for each ABC-XYZ class
# --------------------------------------------------
def get_action(abc, xyz):
    if abc == "A" and xyz == "X":
        return (
            "Tight periodic review; use verified lead time and stock "
            "records for replenishment."
        )
    if abc == "A" and xyz == "Y":
        return (
            "Frequent review; validate demand variability and safety "
            "stock before setting reorder levels."
        )
    if abc == "A" and xyz == "Z":
        return (
            "Priority manual demand review; avoid automatic reorder "
            "decisions until demand spikes are investigated."
        )
    if abc == "A" and xyz == "I":
        return (
            "Manual, order-based replenishment; average daily demand "
            "may be misleading for intermittent sales."
        )
    if abc == "B" and xyz == "X":
        return (
            "Regular periodic review; use actual inventory and "
            "supplier lead-time data."
        )
    if abc == "B" and xyz == "Y":
        return (
            "Periodic review with monitoring of demand variation."
        )
    if abc == "B" and xyz == "Z":
        return (
            "Review variable demand; consider cautious replenishment "
            "with verified stock and lead times."
        )
    if abc == "B" and xyz == "I":
        return (
            "Manual review before replenishment; sales are intermittent."
        )
    if abc == "C" and xyz == "X":
        return (
            "Simple periodic min-max review may be suitable once "
            "stock and lead-time data are available."
        )
    if abc == "C" and xyz == "Y":
        return (
            "Use simple periodic review and monitor changes in demand."
        )
    if abc == "C" and xyz == "Z":
        return (
            "Use cautious replenishment; avoid holding excess stock "
            "based only on a short sales history."
        )
    if abc == "C" and xyz == "I":
        return (
            "Low-cost manual or order-based replenishment; avoid "
            "overstocking slow-moving items."
        )
    return "Manual classification review required."

def get_priority(abc, xyz):
    if abc == "A":
        return "High"
    if abc == "B" and xyz in ["Z", "I"]:
        return "Medium-High"
    if abc == "B":
        return "Medium"
    return "Routine"

inventory["Recommended_Inventory_Action"] = [
    get_action(abc, xyz)
    for abc, xyz in zip(inventory["ABC_Class"], inventory["XYZ_Class"])
]

inventory["Action_Priority"] = [
    get_priority(abc, xyz)
    for abc, xyz in zip(inventory["ABC_Class"], inventory["XYZ_Class"])
]

# --------------------------------------------------
# 2. Keep known data-review issues separate from
# normal inventory action priority
# --------------------------------------------------
inventory["Invoice_581483_Units"] = pd.to_numeric(
    inventory["Invoice_581483_Units"], errors="coerce"
).fillna(0)

inventory["Data_Review_Flag"] = np.where(
    inventory["Invoice_581483_Units"] > 0,
    "Investigate invoice 581483 bulk quantity before trusting demand estimate",
    "No invoice 581483 flag"
)

inventory["Has_Stock_On_Hand_Data"] = (
    inventory["Current_Stock_On_Hand"].notna()
)
inventory["Has_Actual_Lead_Time_Data"] = (
    inventory["Actual_Supplier_Lead_Time_Days"].notna()
)

inventory["Reorder_Quantity_Ready"] = (
    inventory["Has_Stock_On_Hand_Data"]
    & inventory["Has_Actual_Lead_Time_Data"]
)

# Do not create fictional stock quantities or reorder amounts.
inventory["Suggested_Reorder_Quantity"] = pd.NA

# --------------------------------------------------
# 3. Save actionable output and summaries
# --------------------------------------------------
output_path = REPORT_DIR / "inventory_action_recommendations.csv"
priority_path = REPORT_DIR / "inventory_action_priority_summary.csv"
class_path = REPORT_DIR / "inventory_action_class_summary.csv"

inventory.to_csv(output_path, index=False)

priority_summary = (
    inventory.groupby("Action_Priority", as_index=False)
    .agg(
        Products=("StockCode", "nunique"),
        Estimated_30Day_Revenue=("Forecast_30Day_Revenue", "sum")
    )
)
priority_summary["Product_Share_Percent"] = (
    priority_summary["Products"] / len(inventory) * 100
)

class_summary = (
    inventory.groupby(
        ["ABC_Class", "XYZ_Class", "Action_Priority"],
        as_index=False
    )
    .agg(
        Products=("StockCode", "nunique"),
        Estimated_30Day_Revenue=("Forecast_30Day_Revenue", "sum")
    )
)

priority_summary.to_csv(priority_path, index=False)
class_summary.to_csv(class_path, index=False)

print("=" * 70)
print("RETAILPULSE - INVENTORY ACTION RECOMMENDATIONS")
print("=" * 70)
print(f"Products processed: {inventory['StockCode'].nunique():,}")
print(
    "Products with stock-on-hand data: "
    f"{inventory['Has_Stock_On_Hand_Data'].sum():,}"
)
print(
    "Products with actual lead-time data: "
    f"{inventory['Has_Actual_Lead_Time_Data'].sum():,}"
)
print(
    "Products flagged for invoice 581483: "
    f"{(inventory['Invoice_581483_Units'] > 0).sum():,}"
)

print("\nAction priority summary:")
display(priority_summary.round(2))

print("\nABC-XYZ action summary:")
display(class_summary)

print("\nKnown bulk-transaction flag:")
display(
    inventory.loc[
        inventory["Invoice_581483_Units"] > 0,
        [
            "StockCode", "Description", "ABC_XYZ_Class",
            "Invoice_581483_Units", "Data_Review_Flag",
            "Recommended_Inventory_Action"
        ]
    ]
)

print("\nSaved files:")
print(output_path)
print(priority_path)
print(class_path)

print("\nThe master dataset was not modified.")
print(
    "These are proposed policies, not verified reorder quantities. "
    "Real stock-on-hand and actual supplier lead times are still required."
)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\saray\OneDrive\Desktop\Retail")
REPORT_DIR = PROJECT_DIR / "reports" / "inventory"

INPUT_PATH = (
    REPORT_DIR / "product_demand_reorder_baseline_original_stockcodes.csv"
)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Inventory baseline not found: {INPUT_PATH}")

inventory = pd.read_csv(
    INPUT_PATH,
    dtype={"StockCode": "string"}
)

required_columns = [
    "StockCode",
    "Description",
    "Estimated_Reorder_Point_Units",
    "Estimated_Target_Stock_Units",
    "Forecast_30Day_Demand_Units",
    "Forecast_30Day_Revenue",
    "Invoice_581483_Units",
    "Review_Status"
]

missing = [
    col for col in required_columns
    if col not in inventory.columns
]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# --------------------------------------------------
# 1. Clearly labelled planning assumptions
# --------------------------------------------------
# These percentages create demonstration scenarios only.
# They are NOT actual recorded stock quantities.
stock_scenarios = {
    "Illustrative low stock (50% of target)": 0.50,
    "Illustrative target stock (100% of target)": 1.00,
    "Illustrative excess stock (150% of target)": 1.50
}

for col in [
    "Estimated_Reorder_Point_Units",
    "Estimated_Target_Stock_Units",
    "Forecast_30Day_Demand_Units",
    "Forecast_30Day_Revenue",
    "Invoice_581483_Units"
]:
    inventory[col] = pd.to_numeric(inventory[col], errors="coerce")

inventory["Invoice_581483_Units"] = (
    inventory["Invoice_581483_Units"].fillna(0)
)

inventory["Data_Requires_Manual_Review"] = (
    inventory["Invoice_581483_Units"] > 0
)

inventory["Scenario_Data_Type"] = "Illustrative; not actual inventory data"
inventory["Actual_Stock_On_Hand_Available"] = False
inventory["Actual_Lead_Time_Available"] = False

# --------------------------------------------------
# 2. Build the scenarios
# --------------------------------------------------
scenario_rows = []

for _, product in inventory.iterrows():
    target = product["Estimated_Target_Stock_Units"]
    reorder_point = product["Estimated_Reorder_Point_Units"]

    if pd.isna(target) or pd.isna(reorder_point):
        continue

    target = max(0, int(np.ceil(target)))
    reorder_point = max(0, int(np.ceil(reorder_point)))

    for scenario_name, fraction in stock_scenarios.items():
        hypothetical_stock = int(np.floor(target * fraction))

        illustrative_reorder_qty = max(
            target - hypothetical_stock, 0
        )
        illustrative_excess_qty = max(
            hypothetical_stock - target, 0
        )

        if product["Data_Requires_Manual_Review"]:
            status = "Manual review required; do not use for automatic ordering"
        elif hypothetical_stock < reorder_point:
            status = "Illustrative: below reorder point"
        elif hypothetical_stock < target:
            status = "Illustrative: below target stock"
        elif hypothetical_stock == target:
            status = "Illustrative: at target stock"
        else:
            status = "Illustrative: above target stock"

        scenario_rows.append({
            "StockCode": product["StockCode"],
            "Description": product["Description"],
            "Scenario": scenario_name,
            "Assumed_Stock_Fraction_Of_Target": fraction,
            "Hypothetical_Stock_On_Hand_Units": hypothetical_stock,
            "Estimated_Reorder_Point_Units": reorder_point,
            "Estimated_Target_Stock_Units": target,
            "Illustrative_Reorder_Quantity_Units": illustrative_reorder_qty,
            "Illustrative_Excess_Stock_Units": illustrative_excess_qty,
            "Forecast_30Day_Demand_Units": product["Forecast_30Day_Demand_Units"],
            "Forecast_30Day_Revenue": product["Forecast_30Day_Revenue"],
            "Data_Requires_Manual_Review": product["Data_Requires_Manual_Review"],
            "Scenario_Status": status,
            "Operational_Order_Approved": False,
            "Scenario_Data_Type": "Illustrative; not actual inventory data"
        })

scenarios_df = pd.DataFrame(scenario_rows)

# --------------------------------------------------
# 3. Save detailed scenarios and summary tables
# --------------------------------------------------
scenario_path = REPORT_DIR / "inventory_illustrative_reorder_scenarios.csv"
summary_path = REPORT_DIR / "inventory_illustrative_scenario_summary.csv"
bulk_review_path = REPORT_DIR / "inventory_bulk_transaction_manual_review.csv"

scenarios_df.to_csv(scenario_path, index=False)

# Summaries deliberately keep counts and hypothetical units labelled.
summary = (
    scenarios_df.groupby("Scenario", as_index=False)
    .agg(
        Products=("StockCode", "nunique"),
        Illustrative_Reorder_Units=("Illustrative_Reorder_Quantity_Units", "sum"),
        Illustrative_Excess_Units=("Illustrative_Excess_Stock_Units", "sum"),
        Products_Flagged_For_Manual_Review=("Data_Requires_Manual_Review", "sum")
    )
)

summary["Scenario_Data_Type"] = "Illustrative; not actual inventory data"
summary.to_csv(summary_path, index=False)

bulk_review = scenarios_df[
    scenarios_df["Data_Requires_Manual_Review"]
].copy()

bulk_review.to_csv(bulk_review_path, index=False)

print("=" * 70)
print("RETAILPULSE - ILLUSTRATIVE INVENTORY REORDER SCENARIOS")
print("=" * 70)

print("Products:", inventory["StockCode"].nunique())
print("Scenario records:", len(scenarios_df))
print(
    "Products flagged for manual review:",
    inventory["Data_Requires_Manual_Review"].sum()
)

print("\nScenario summary (hypothetical quantities only):")
display(summary)

print("\nExample scenarios for PAPER CRAFT LITTLE BIRDIE (23843):")
display(
    scenarios_df[
        scenarios_df["StockCode"].eq("23843")
    ][[
        "Scenario",
        "Hypothetical_Stock_On_Hand_Units",
        "Estimated_Reorder_Point_Units",
        "Estimated_Target_Stock_Units",
        "Illustrative_Reorder_Quantity_Units",
        "Scenario_Status",
        "Operational_Order_Approved"
    ]]
)

print("\nSaved files:")
print(scenario_path)
print(summary_path)
print(bulk_review_path)

print("\nImportant:")
print("- All stock quantities in this scenario are hypothetical.")
print("- No actual purchase order or reorder approval is generated.")
print("- The master dataset was not modified.")




In [ ]:
# ============================================================
# RETAILPULSE — INVENTORY OPTIMIZATION BACKTEST
# Historical simulation: compare stock replenishment policies
# ============================================================

from pathlib import Path
import math
import numpy as np
import pandas as pd

# Reuse the project paths already defined in the previous cell.
SOURCE_PATH = PROJECT_DIR / "data" / "raw" / "online_retail_II.xlsx"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not SOURCE_PATH.exists():
    raise FileNotFoundError(
        f"Source workbook not found: {SOURCE_PATH}"
    )

# ------------------------------------------------------------
# 1. Simulation assumptions
# ------------------------------------------------------------

WINDOW_DAYS = 90
TRAIN_DAYS = 60
TEST_DAYS = WINDOW_DAYS - TRAIN_DAYS

LEAD_TIME_DAYS = 7        # Assumed, not an observed supplier lead time
REVIEW_PERIOD_DAYS = 1    # Simulate a daily inventory review

POLICIES = {
    "No safety buffer": 0.0000,
    "Low buffer": 0.5000,
    "Moderate buffer": 0.8400,
    "Higher buffer": 1.0400,
    "90th-percentile buffer": 1.2816,
    "95th-percentile buffer": 1.6449,
}

# ------------------------------------------------------------
# 2. Load original transactions
#    Preserve original StockCode values and suffixes.
#    Do not use the three helper flags.
# ------------------------------------------------------------

required = ["StockCode", "Quantity", "InvoiceDate", "Price"]
sheets = ["Year 2009-2010", "Year 2010-2011"]

parts = []

for sheet in sheets:
    print(f"Reading: {sheet}")

    part = pd.read_excel(
        SOURCE_PATH,
        sheet_name=sheet,
        usecols=required,
        converters={
            "StockCode": lambda x: str(x).strip()
            if pd.notna(x) else np.nan
        },
    )
    parts.append(part)

sales = pd.concat(parts, ignore_index=True)

sales["StockCode"] = sales["StockCode"].astype("string").str.strip()
sales["Quantity"] = pd.to_numeric(sales["Quantity"], errors="coerce")
sales["Price"] = pd.to_numeric(sales["Price"], errors="coerce")
sales["InvoiceDate"] = pd.to_datetime(
    sales["InvoiceDate"], errors="coerce"
)

# Use positive-quantity, positive-price transactions as recorded demand.
# Transaction lines are not silently deduplicated.
sales = sales.loc[
    sales["StockCode"].notna()
    & sales["StockCode"].ne("")
    & sales["InvoiceDate"].notna()
    & sales["Quantity"].gt(0)
    & sales["Price"].gt(0),
    required,
].copy()

sales["InvoiceDate"] = sales["InvoiceDate"].dt.normalize()
sales["RevenueProxy"] = sales["Quantity"] * sales["Price"]

if sales.empty:
    raise ValueError("No eligible positive sales transactions were found.")

# ------------------------------------------------------------
# 3. Define the historical train/test window
# ------------------------------------------------------------

latest_date = sales["InvoiceDate"].max()
window_start = latest_date - pd.Timedelta(days=WINDOW_DAYS - 1)

dates = pd.date_range(window_start, latest_date, freq="D")
train_dates = dates[:TRAIN_DAYS]
test_dates = dates[TRAIN_DAYS:]

recent = sales.loc[
    sales["InvoiceDate"].between(window_start, latest_date)
].copy()

daily_units = (
    recent.pivot_table(
        index="InvoiceDate",
        columns="StockCode",
        values="Quantity",
        aggfunc="sum",
        fill_value=0,
    )
    .reindex(index=dates, fill_value=0)
    .fillna(0)
)

daily_units.columns = daily_units.columns.astype(str)

train_daily = daily_units.loc[train_dates]
test_daily = daily_units.loc[test_dates]

train_revenue = (
    sales.loc[sales["InvoiceDate"].isin(train_dates)]
    .groupby("StockCode")["RevenueProxy"]
    .sum()
)
train_revenue.index = train_revenue.index.astype(str)
train_revenue = train_revenue.reindex(daily_units.columns, fill_value=0)

print("\nHistorical backtest window")
print(f"Window: {dates[0].date()} to {dates[-1].date()}")
print(f"Training: {train_dates[0].date()} to {train_dates[-1].date()}")
print(f"Testing:  {test_dates[0].date()} to {test_dates[-1].date()}")
print(f"Eligible transaction rows: {len(recent):,}")
print(f"Products simulated: {len(daily_units.columns):,}")

# ------------------------------------------------------------
# 4. Calculate ABC-XYZ classes using training data only
# ------------------------------------------------------------

sorted_revenue = train_revenue.sort_values(ascending=False)
total_training_revenue = sorted_revenue.sum()

if total_training_revenue > 0:
    cumulative_share = sorted_revenue.cumsum() / total_training_revenue
    abc_map = {
        str(code): (
            "A" if share <= 0.80
            else "B" if share <= 0.95
            else "C"
        )
        for code, share in cumulative_share.items()
    }
else:
    abc_map = {str(code): "C" for code in daily_units.columns}

xyz_map = {}
profile_rows = []

for code in daily_units.columns:
    history = train_daily[code].to_numpy(dtype=float)
    future = test_daily[code].to_numpy(dtype=float)

    mean_units = float(np.mean(history))
    std_units = float(np.std(history, ddof=1)) if len(history) > 1 else 0.0
    sales_days = int(np.sum(history > 0))
    cv = std_units / mean_units if mean_units > 0 else np.nan

    if sales_days < 5:
        xyz = "I"
    elif pd.notna(cv) and cv <= 0.5:
        xyz = "X"
    elif pd.notna(cv) and cv <= 1.0:
        xyz = "Y"
    else:
        xyz = "Z"

    xyz_map[str(code)] = xyz

    profile_rows.append({
        "StockCode": str(code),
        "ABC_Class": abc_map.get(str(code), "C"),
        "XYZ_Class": xyz,
        "Training_Revenue": float(train_revenue.get(code, 0)),
        "Training_Daily_Mean_Units": mean_units,
        "Training_Daily_Std_Units": std_units,
        "Training_Sales_Days": sales_days,
        "Training_CV": cv,
        "Test_Demand_Units": float(future.sum()),
        "Cold_Start_In_Test": bool(mean_units == 0 and future.sum() > 0),
    })

profile = pd.DataFrame(profile_rows)

# Carry forward the manual review flag from the existing baseline if available.
baseline_path = REPORT_DIR / "product_demand_reorder_baseline_original_stockcodes.csv"

if baseline_path.exists():
    baseline_flags = pd.read_csv(
        baseline_path, dtype={"StockCode": "string"}
    )
    if {"StockCode", "Invoice_581483_Units"}.issubset(baseline_flags.columns):
        baseline_flags["StockCode"] = (
            baseline_flags["StockCode"].astype("string").str.strip()
        )
        review_codes = set(
            baseline_flags.loc[
                pd.to_numeric(
                    baseline_flags["Invoice_581483_Units"], errors="coerce"
                ).fillna(0).gt(0),
                "StockCode",
            ].dropna().astype(str)
        )
    else:
        review_codes = set()
else:
    review_codes = set()

profile["Manual_Review_Flag"] = profile["StockCode"].isin(review_codes)

# ------------------------------------------------------------
# 5. Simulate one inventory policy for one product
# ------------------------------------------------------------

def simulate_inventory_policy(history, future, z):
    mean_units = float(np.mean(history)) if len(history) else 0.0
    std_units = float(np.std(history, ddof=1)) if len(history) > 1 else 0.0

    mean_units = max(0.0, mean_units) if np.isfinite(mean_units) else 0.0
    std_units = max(0.0, std_units) if np.isfinite(std_units) else 0.0

    reorder_point = math.ceil(
        mean_units * LEAD_TIME_DAYS
        + z * std_units * math.sqrt(LEAD_TIME_DAYS)
    )

    target_stock = math.ceil(
        mean_units * (LEAD_TIME_DAYS + REVIEW_PERIOD_DAYS)
        + z * std_units * math.sqrt(
            LEAD_TIME_DAYS + REVIEW_PERIOD_DAYS
        )
    )
    target_stock = max(reorder_point, target_stock, 0)
    reorder_point = max(reorder_point, 0)

    # Opening stock is simulated at the policy's target.
    # It is not actual stock-on-hand from the retailer.
    on_hand = float(target_stock)
    pipeline = {}

    total_demand = 0.0
    fulfilled = 0.0
    unmet = 0.0
    stockout_days = 0
    ending_stock_sum = 0.0
    total_ordered = 0.0
    cover_surplus = 0.0

    normal_cover_threshold = mean_units * (
        LEAD_TIME_DAYS + REVIEW_PERIOD_DAYS
    )

    for day_index, demand_value in enumerate(future):
        received = pipeline.pop(day_index, 0.0)
        on_hand += received

        demand = max(0.0, float(demand_value))
        sold = min(on_hand, demand)
        lost = max(0.0, demand - sold)
        on_hand -= sold

        total_demand += demand
        fulfilled += sold
        unmet += lost

        if lost > 0:
            stockout_days += 1

        ending_stock_sum += on_hand
        cover_surplus += max(0.0, on_hand - normal_cover_threshold)

        inventory_position = on_hand + sum(pipeline.values())

        if inventory_position <= reorder_point:
            order_qty = max(0.0, target_stock - inventory_position)

            if order_qty > 0:
                arrival_day = day_index + LEAD_TIME_DAYS
                pipeline[arrival_day] = (
                    pipeline.get(arrival_day, 0.0) + order_qty
                )
                total_ordered += order_qty

    fill_rate = (
        fulfilled / total_demand * 100
        if total_demand > 0 else np.nan
    )

    return {
        "Training_Daily_Mean_Units": mean_units,
        "Reorder_Point_Units": reorder_point,
        "Target_Stock_Units": target_stock,
        "Test_Demand_Units": total_demand,
        "Fulfilled_Units": fulfilled,
        "Unmet_Demand_Units": unmet,
        "Fill_Rate_Pct": fill_rate,
        "Stockout_Days": stockout_days,
        "Average_On_Hand_Units": ending_stock_sum / max(len(future), 1),
        "Ending_On_Hand_Units": on_hand,
        "Units_Ordered_During_Test": total_ordered,
        "Cover_Surplus_Unit_Days_Proxy": cover_surplus,
    }

# ------------------------------------------------------------
# 6. Run the policy comparison
# ------------------------------------------------------------

metric_rows = []

for policy_name, z in POLICIES.items():
    print(f"Simulating: {policy_name}")

    for code in daily_units.columns:
        result = simulate_inventory_policy(
            train_daily[code].to_numpy(dtype=float),
            test_daily[code].to_numpy(dtype=float),
            z,
        )

        sku_profile = profile.loc[profile["StockCode"].eq(str(code))].iloc[0]

        metric_rows.append({
            "Scenario": policy_name,
            "Safety_Factor_Z": z,
            "StockCode": str(code),
            "ABC_Class": sku_profile["ABC_Class"],
            "XYZ_Class": sku_profile["XYZ_Class"],
            "Cold_Start_In_Test": sku_profile["Cold_Start_In_Test"],
            "Manual_Review_Flag": sku_profile["Manual_Review_Flag"],
            **result,
        })

sku_metrics = pd.DataFrame(metric_rows)

# ------------------------------------------------------------
# 7. Summarize overall performance and known-history products
# ------------------------------------------------------------

def safe_reduction_pct(baseline, current):
    if baseline == 0:
        return 0.0 if current == 0 else np.nan
    return (baseline - current) / baseline * 100.0


summary_rows = []

cohorts = {
    "All products in test window": pd.Series(True, index=profile.index),
    "Products with training-period demand": profile[
        "Training_Daily_Mean_Units"
    ].gt(0),
}

profile_index = profile.set_index("StockCode")

for cohort_name in cohorts:
    if cohort_name == "All products in test window":
        included_codes = set(profile["StockCode"])
    else:
        included_codes = set(
            profile.loc[
                profile["Training_Daily_Mean_Units"].gt(0), "StockCode"
            ]
        )

    for policy_name, z in POLICIES.items():
        part = sku_metrics[
            sku_metrics["Scenario"].eq(policy_name)
            & sku_metrics["StockCode"].isin(included_codes)
        ]

        demand = part["Test_Demand_Units"].sum()
        fulfilled = part["Fulfilled_Units"].sum()
        unmet = part["Unmet_Demand_Units"].sum()

        summary_rows.append({
            "Population": cohort_name,
            "Scenario": policy_name,
            "Safety_Factor_Z": z,
            "Products": part["StockCode"].nunique(),
            "Test_Demand_Units": demand,
            "Fulfilled_Units": fulfilled,
            "Unmet_Demand_Units": unmet,
            "Fill_Rate_Pct": fulfilled / demand * 100 if demand > 0 else np.nan,
            "Stockout_SKU_Days": part["Stockout_Days"].sum(),
            "Portfolio_Average_On_Hand_Units": part[
                "Average_On_Hand_Units"
            ].sum(),
            "Units_Ordered_During_Test": part[
                "Units_Ordered_During_Test"
            ].sum(),
            "Cover_Surplus_Unit_Days_Proxy": part[
                "Cover_Surplus_Unit_Days_Proxy"
            ].sum(),
            "Cold_Start_SKUs": part.loc[
                part["Cold_Start_In_Test"], "StockCode"
            ].nunique(),
        })

policy_summary = pd.DataFrame(summary_rows)

# Compare each policy against the no-safety-buffer policy for the same population.
policy_summary["Unmet_Demand_Reduction_Pct_vs_No_Buffer"] = np.nan
policy_summary["Stockout_Day_Reduction_Pct_vs_No_Buffer"] = np.nan
policy_summary["Average_On_Hand_Change_Pct_vs_No_Buffer"] = np.nan

for population in policy_summary["Population"].unique():
    population_mask = policy_summary["Population"].eq(population)
    base_rows = policy_summary[
        population_mask
        & policy_summary["Scenario"].eq("No safety buffer")
    ]

    if base_rows.empty:
        continue

    base = base_rows.iloc[0]

    for idx in policy_summary.index[population_mask]:
        row = policy_summary.loc[idx]

        policy_summary.loc[
            idx, "Unmet_Demand_Reduction_Pct_vs_No_Buffer"
        ] = safe_reduction_pct(
            float(base["Unmet_Demand_Units"]),
            float(row["Unmet_Demand_Units"]),
        )

        policy_summary.loc[
            idx, "Stockout_Day_Reduction_Pct_vs_No_Buffer"
        ] = safe_reduction_pct(
            float(base["Stockout_SKU_Days"]),
            float(row["Stockout_SKU_Days"]),
        )

        base_stock = float(base["Portfolio_Average_On_Hand_Units"])
        current_stock = float(row["Portfolio_Average_On_Hand_Units"])

        policy_summary.loc[
            idx, "Average_On_Hand_Change_Pct_vs_No_Buffer"
        ] = (
            (current_stock - base_stock) / base_stock * 100
            if base_stock > 0 else np.nan
        )

# ------------------------------------------------------------
# 8. ABC-XYZ results and save all outputs
# ------------------------------------------------------------

abc_xyz_summary = (
    sku_metrics.groupby(
        ["Scenario", "ABC_Class", "XYZ_Class"],
        dropna=False,
        as_index=False,
    )
    .agg(
        Products=("StockCode", "nunique"),
        Test_Demand_Units=("Test_Demand_Units", "sum"),
        Fulfilled_Units=("Fulfilled_Units", "sum"),
        Unmet_Demand_Units=("Unmet_Demand_Units", "sum"),
        Stockout_SKU_Days=("Stockout_Days", "sum"),
        Units_Ordered=("Units_Ordered_During_Test", "sum"),
    )
)

abc_xyz_summary["Fill_Rate_Pct"] = np.where(
    abc_xyz_summary["Test_Demand_Units"] > 0,
    abc_xyz_summary["Fulfilled_Units"]
    / abc_xyz_summary["Test_Demand_Units"] * 100,
    np.nan,
)

sku_metrics_path = REPORT_DIR / "inventory_backtest_sku_metrics.csv"
summary_path = REPORT_DIR / "inventory_backtest_policy_summary.csv"
profile_path = REPORT_DIR / "inventory_backtest_abc_xyz_profile.csv"
abc_xyz_path = REPORT_DIR / "inventory_backtest_abc_xyz_summary.csv"
assumptions_path = REPORT_DIR / "inventory_backtest_assumptions.md"

sku_metrics.to_csv(sku_metrics_path, index=False)
policy_summary.to_csv(summary_path, index=False)
profile.to_csv(profile_path, index=False)
abc_xyz_summary.to_csv(abc_xyz_path, index=False)

assumptions_text = f"""# RetailPulse Inventory Backtest

## Design
- Source: `data/raw/online_retail_II.xlsx`
- Window: {dates[0].date()} to {dates[-1].date()}
- Training period: {train_dates[0].date()} to {train_dates[-1].date()}
- Test period: {test_dates[0].date()} to {test_dates[-1].date()}
- Assumed supplier lead time: {LEAD_TIME_DAYS} days
- Assumed inventory review period: {REVIEW_PERIOD_DAYS} day
- Policies compared: {", ".join(POLICIES.keys())}
- Product codes retain their original suffixes.
- Only positive-quantity, positive-price transactions with valid dates are included.
- The three helper flags are not used. Transaction lines are not silently deduplicated.

## Interpretation
- Fill rate, unmet units, and stockout SKU-days are measured in this simulation.
- Opening inventory and lead time are assumptions, not actual operating records.
- Cover surplus is only a proxy for excess inventory; some buffer stock may be intentional.
- Results are sensitive to intermittent demand and unusual transaction spikes.
- The no-safety-buffer policy is a simulated comparison baseline, not proof of actual current operations.
- Products without training-period demand are also reported separately because their future demand cannot be inferred by this simple policy.

## Important limitation
This historical simulation can compare policies under stated assumptions. It cannot establish a real-world 25–40% reduction in overstock or understock without observed inventory levels, actual lead times, replenishment records, and stockout data.
"""

assumptions_path.write_text(assumptions_text, encoding="utf-8")

# ------------------------------------------------------------
# 9. Display the results
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("RETAILPULSE — INVENTORY POLICY BACKTEST COMPLETE")
print("=" * 75)

display_columns = [
    "Population",
    "Scenario",
    "Products",
    "Fill_Rate_Pct",
    "Unmet_Demand_Units",
    "Unmet_Demand_Reduction_Pct_vs_No_Buffer",
    "Stockout_SKU_Days",
    "Stockout_Day_Reduction_Pct_vs_No_Buffer",
    "Portfolio_Average_On_Hand_Units",
    "Average_On_Hand_Change_Pct_vs_No_Buffer",
]

display(policy_summary[display_columns].round(2))

print("\nABC-XYZ policy results:")
display(abc_xyz_summary.round(2))

print("\nSaved outputs:")
for path in [
    sku_metrics_path,
    summary_path,
    profile_path,
    abc_xyz_path,
    assumptions_path,
]:
    print(path)

print("\nImportant:")
print("- These metrics describe a historical simulation, not real warehouse operations.")
print("- Do not mark the operational 25–40% target as validated based on this alone.")
print("- Manual-review products remain included and flagged, not silently removed.")

In [ ]:

from pathlib import Path
import pandas as pd

# Find the project folder automatically
current_dir = Path.cwd()
project_root = next(
    (
        folder for folder in [current_dir, *current_dir.parents]
        if (folder / "data" / "raw" / "online_retail_ii.xlsx").exists()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        f"Could not locate online_retail_ii.xlsx from {current_dir}.\n"
        "Please check the actual location of your Retail project folder."
    )

workbook_path = (
    project_root / "data" / "raw" / "online_retail_ii.xlsx"
)

print("Project folder:", project_root)
print("Workbook found:", workbook_path)

xls = pd.ExcelFile(workbook_path)
print("\nWorkbook sheets:", xls.sheet_names)

for sheet in xls.sheet_names:
    df = pd.read_excel(workbook_path, sheet_name=sheet)

    print(f"\nSheet: {sheet}")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if "StockCode" in df.columns:
        print("Unique original StockCodes:", df["StockCode"].nunique())

    if "InvoiceDate" in df.columns:
        dates = pd.to_datetime(df["InvoiceDate"], errors="coerce")
        print("Date range:", dates.min(), "to", dates.max())


In [ ]:

from pathlib import Path
import pandas as pd

workbook_path = (
    Path("d:/Retail/data/raw/online_retail_ii.xlsx")
)

# The final 90-day historical window
start_date = pd.Timestamp("2011-09-11")
end_date = pd.Timestamp("2011-12-10")  # exclusive

# Load the 2010–2011 sheet
df = pd.read_excel(
    workbook_path,
    sheet_name="Year 2010-2011"
)

# Standardize only the working copy's date/numeric columns
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)
df["Quantity"] = pd.to_numeric(
    df["Quantity"], errors="coerce"
)
df["Price"] = pd.to_numeric(
    df["Price"], errors="coerce"
)

# Apply the date window
window = df.loc[
    df["InvoiceDate"].ge(start_date)
    & df["InvoiceDate"].lt(end_date)
].copy()

print("Date-window rows:", len(window))

# Show the impact of the inventory eligibility rules
eligible = window.loc[
    window["StockCode"].notna()
    & window["InvoiceDate"].notna()
    & window["Quantity"].gt(0)
    & window["Price"].gt(0)
].copy()

eligible["Revenue"] = eligible["Quantity"] * eligible["Price"]

print("\n--- Eligible inventory demand ---")
print("Eligible rows:", len(eligible))
print("Unique original StockCodes:", eligible["StockCode"].nunique())
print("Total units:", eligible["Quantity"].sum())
print("Total revenue:", round(eligible["Revenue"].sum(), 2))
print(
    "Date range:",
    eligible["InvoiceDate"].min(),
    "to",
    eligible["InvoiceDate"].max()
)

# Audit the previously noted high-volume invoice line
print("\n--- Invoice 581483 audit ---")
invoice_audit = window.loc[
    window["Invoice"].astype(str).str.strip().eq("581483"),
    [
        "Invoice", "StockCode", "Description",
        "Quantity", "InvoiceDate", "Price", "Customer ID"
    ]
]
print(invoice_audit.to_string(index=False))


In [ ]:

# Use the `eligible` DataFrame from the previous cell.

outlier_mask = (
    eligible["Invoice"].astype(str).str.strip().eq("581483")
    & eligible["StockCode"].astype(str).str.strip().eq("23843")
)

outlier_rows = eligible.loc[outlier_mask]

print("Matching invoice lines:", len(outlier_rows))

if not outlier_rows.empty:
    outlier_units = outlier_rows["Quantity"].sum()
    outlier_revenue = outlier_rows["Revenue"].sum()

    total_units = eligible["Quantity"].sum()
    total_revenue = eligible["Revenue"].sum()

    print("\n--- Outlier impact ---")
    print("Outlier units:", int(outlier_units))
    print("Outlier revenue:", round(outlier_revenue, 2))
    print("Share of eligible units (%):",
          round(outlier_units / total_units * 100, 2))
    print("Share of eligible revenue (%):",
          round(outlier_revenue / total_revenue * 100, 2))

    without_line = eligible.loc[~outlier_mask]

    print("\n--- Totals excluding this line (diagnostic only) ---")
    print("Rows:", len(without_line))
    print("Unique StockCodes:", without_line["StockCode"].nunique())
    print("Units:", int(without_line["Quantity"].sum()))
    print("Revenue:", round(without_line["Revenue"].sum(), 2))

    print(
        "\nImportant: these exclusion totals are for comparison only. "
        "The original data remains unchanged."
    )
else:
    print("No matching eligible invoice line was found.")


In [ ]:

import numpy as np
import pandas as pd

# Keep the full eligible dataset, including unusual bulk transactions.
inventory_data = eligible.copy()

# Calendar-day demand for each original StockCode
inventory_data["Date"] = inventory_data["InvoiceDate"].dt.normalize()

daily_demand = (
    inventory_data.groupby(["StockCode", "Date"], as_index=False)
    ["Quantity"].sum()
)

# Include zero-demand days in the daily demand statistics.
all_dates = pd.date_range(
    start="2011-09-11",
    end="2011-12-09",
    freq="D"
)

products = inventory_data["StockCode"].drop_duplicates()

full_index = pd.MultiIndex.from_product(
    [products, all_dates],
    names=["StockCode", "Date"]
)

daily_complete = (
    daily_demand.set_index(["StockCode", "Date"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

product_baseline = (
    daily_complete.groupby("StockCode")
    .agg(
        Total_Units=("Quantity", "sum"),
        Active_Days=("Quantity", lambda x: int((x > 0).sum())),
        Avg_Daily_Demand=("Quantity", "mean"),
        Daily_Demand_Std=("Quantity", "std"),
        Max_Daily_Demand=("Quantity", "max")
    )
    .reset_index()
)

product_baseline["Daily_Demand_Std"] = (
    product_baseline["Daily_Demand_Std"].fillna(0)
)

# ABC classification based on total units sold in the window.
product_baseline = product_baseline.sort_values(
    "Total_Units", ascending=False
).reset_index(drop=True)

total_units = product_baseline["Total_Units"].sum()

product_baseline["Cumulative_Unit_Share"] = (
    product_baseline["Total_Units"].cumsum() / total_units
)

product_baseline["ABC_Class"] = np.select(
    [
        product_baseline["Cumulative_Unit_Share"] <= 0.80,
        product_baseline["Cumulative_Unit_Share"] <= 0.95
    ],
    ["A", "B"],
    default="C"
)

# Save a separate derived file; do not overwrite the master CSV.
output_dir = Path("d:/Retail/reports/inventory")
output_dir.mkdir(parents=True, exist_ok=True)

baseline_path = output_dir / "product_demand_baseline_step4.csv"
product_baseline.to_csv(baseline_path, index=False)

print("Products in baseline:", len(product_baseline))
print("Total units:", int(product_baseline["Total_Units"].sum()))
print("\nABC product counts:")
print(product_baseline["ABC_Class"].value_counts().sort_index())

print("\nTop 10 products by units:")
print(
    product_baseline[
        [
            "StockCode", "Total_Units", "Active_Days",
            "Avg_Daily_Demand", "Daily_Demand_Std",
            "ABC_Class"
        ]
    ].head(10).to_string(index=False)
)

print("\nSaved baseline:", baseline_path)


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

# Use the verified Step 4 baseline.
baseline = product_baseline.copy()

# Explicit modeling assumptions
LEAD_TIME_DAYS = 7
SERVICE_LEVEL = 0.90
Z_SCORE = 1.2816  # Approximate one-sided 90% service-level factor

# Expected demand during lead time
baseline["Lead_Time_Demand"] = (
    baseline["Avg_Daily_Demand"] * LEAD_TIME_DAYS
)

# Safety stock under the assumption of independent daily demand
baseline["Safety_Stock"] = (
    Z_SCORE
    * baseline["Daily_Demand_Std"]
    * np.sqrt(LEAD_TIME_DAYS)
).round().astype(int)

# Reorder point = expected lead-time demand + safety stock
baseline["Reorder_Point"] = np.ceil(
    baseline["Lead_Time_Demand"] + baseline["Safety_Stock"]
).astype(int)

# A simple initial order-up-to suggestion:
# cover lead time plus 7 additional days of average demand.
baseline["Suggested_Order_Up_To"] = np.ceil(
    baseline["Lead_Time_Demand"]
    + baseline["Safety_Stock"]
    + baseline["Avg_Daily_Demand"] * 7
).astype(int)

# Flag unusually concentrated demand for review.
baseline["Demand_Review_Flag"] = np.where(
    (baseline["Active_Days"] <= 2)
    | (
        baseline["Daily_Demand_Std"]
        > 3 * baseline["Avg_Daily_Demand"]
    ),
    "Review demand pattern",
    "Standard review"
)

# Save as a separate derived output.
output_dir = Path("d:/Retail/reports/inventory")
output_dir.mkdir(parents=True, exist_ok=True)

rop_path = output_dir / "inventory_reorder_points_step5.csv"
baseline.to_csv(rop_path, index=False)

print("Products with reorder points:", len(baseline))
print("Assumed lead time (days):", LEAD_TIME_DAYS)
print("Assumed service level (%):", SERVICE_LEVEL * 100)
print("\nProducts flagged for demand review:",
      (baseline["Demand_Review_Flag"] == "Review demand pattern").sum())

print("\nTop 10 products by total historical units:")
print(
    baseline.sort_values("Total_Units", ascending=False)[
        [
            "StockCode",
            "Total_Units",
            "Avg_Daily_Demand",
            "Daily_Demand_Std",
            "Safety_Stock",
            "Reorder_Point",
            "Suggested_Order_Up_To",
            "Demand_Review_Flag"
        ]
    ].head(10).to_string(index=False)
)

print("\nSaved reorder-point file:", rop_path)


In [ ]:

from pathlib import Path
import pandas as pd

rop_path = Path(
    "d:/Retail/reports/inventory/inventory_reorder_points_step5.csv"
)

if not rop_path.exists():
    raise FileNotFoundError(f"Saved output not found: {rop_path}")

check = pd.read_csv(rop_path)

required_columns = [
    "StockCode",
    "Total_Units",
    "Active_Days",
    "Avg_Daily_Demand",
    "Daily_Demand_Std",
    "Safety_Stock",
    "Reorder_Point",
    "Suggested_Order_Up_To",
    "Demand_Review_Flag",
]

missing = [col for col in required_columns if col not in check.columns]

print("File exists:", rop_path.exists())
print("File size (KB):", round(rop_path.stat().st_size / 1024, 2))
print("Rows:", len(check))
print("Unique StockCodes:", check["StockCode"].nunique())
print("Missing required columns:", missing)

print("\nMissing values in key calculations:")
print(
    check[
        [
            "Avg_Daily_Demand",
            "Daily_Demand_Std",
            "Safety_Stock",
            "Reorder_Point",
            "Suggested_Order_Up_To",
        ]
    ].isna().sum()
)

print("\nInvalid reorder points:",
      (check["Reorder_Point"] < 0).sum())

print("\nProducts flagged for demand review:")
print(check["Demand_Review_Flag"].value_counts(dropna=False))

print("\nProducts with the largest safety stock:")
print(
    check.sort_values("Safety_Stock", ascending=False)[
        [
            "StockCode",
            "Total_Units",
            "Active_Days",
            "Safety_Stock",
            "Reorder_Point",
            "Demand_Review_Flag",
        ]
    ].head(10).to_string(index=False)
)


In [ ]:


from pathlib import Path
import numpy as np
import pandas as pd

source_path = Path(
    "d:/Retail/reports/inventory/inventory_reorder_points_step5.csv"
)
output_path = Path(
    "d:/Retail/reports/inventory/inventory_reorder_points_step7.csv"
)

df = pd.read_csv(source_path)

# Historical window is 90 days; retain the existing 7-day lead-time assumption.
LEAD_TIME_DAYS = 7
Z_SCORE = 1.2816

# Baseline estimate from the existing calculation.
df["Original_Safety_Stock"] = df["Safety_Stock"]

# Diagnostic demand cap:
# Limit the daily demand used for variability to the 95th percentile
# of a simple Poisson-style demand proxy. This is a sensitivity analysis,
# not a statistically validated demand model.
daily_mean = df["Avg_Daily_Demand"].clip(lower=0)
poisson_std = np.sqrt(daily_mean)

# Use the smaller of observed daily variability and this proxy.
# This reduces the effect of extreme spikes, but requires backtesting.
df["Diagnostic_Demand_Std"] = np.minimum(
    df["Daily_Demand_Std"].fillna(0),
    poisson_std
)

df["Diagnostic_Safety_Stock"] = np.ceil(
    Z_SCORE * df["Diagnostic_Demand_Std"] * np.sqrt(LEAD_TIME_DAYS)
).astype(int)

df["Diagnostic_Reorder_Point"] = np.ceil(
    daily_mean * LEAD_TIME_DAYS + df["Diagnostic_Safety_Stock"]
).astype(int)

df["Safety_Stock_Change"] = (
    df["Original_Safety_Stock"] - df["Diagnostic_Safety_Stock"]
)

df.to_csv(output_path, index=False)

print("Products compared:", len(df))
print("Diagnostic file saved:", output_path)

print("\nSafety-stock comparison:")
print(
    df[
        [
            "StockCode",
            "Active_Days",
            "Avg_Daily_Demand",
            "Original_Safety_Stock",
            "Diagnostic_Safety_Stock",
            "Diagnostic_Reorder_Point",
            "Safety_Stock_Change",
        ]
    ]
    .sort_values("Original_Safety_Stock", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\nProducts where diagnostic safety stock is lower:",
      (df["Diagnostic_Safety_Stock"] < df["Original_Safety_Stock"]).sum())

print(
    "\nImportant: diagnostic estimates are not approved reorder settings "
    "until compared with the historical backtest."
)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

workbook_path = Path(
    "d:/Retail/data/raw/online_retail_ii.xlsx"
)
recommendations_path = Path(
    "d:/Retail/reports/inventory/inventory_reorder_points_step7.csv"
)
output_dir = Path("d:/Retail/reports/inventory")
output_dir.mkdir(parents=True, exist_ok=True)

# Load original source transactions
source = pd.read_excel(
    workbook_path,
    sheet_name="Year 2010-2011"
)

source["InvoiceDate"] = pd.to_datetime(
    source["InvoiceDate"], errors="coerce"
)
source["Quantity"] = pd.to_numeric(
    source["Quantity"], errors="coerce"
)
source["Price"] = pd.to_numeric(
    source["Price"], errors="coerce"
)

source = source.loc[
    source["StockCode"].notna()
    & source["InvoiceDate"].notna()
    & source["Quantity"].gt(0)
    & source["Price"].gt(0)
].copy()

source["Date"] = source["InvoiceDate"].dt.normalize()

# Same historical 90-day window used in previous steps
source = source.loc[
    source["Date"].between(
        pd.Timestamp("2011-09-11"),
        pd.Timestamp("2011-12-09")
    )
].copy()

# Fixed chronological comparison:
# first 60 days for training; final 30 days for evaluation.
train_end = pd.Timestamp("2011-11-09")
test_start = pd.Timestamp("2011-11-10")
test_end = pd.Timestamp("2011-12-09")

train = source.loc[source["Date"] <= train_end]
test = source.loc[
    source["Date"].between(test_start, test_end)
]

print("Training rows:", len(train))
print("Test rows:", len(test))
print("Training dates:", train["Date"].min(), "to", train["Date"].max())
print("Test dates:", test["Date"].min(), "to", test["Date"].max())

# Load two candidate reorder policies
policies = pd.read_csv(recommendations_path)

# Training-period daily demand statistics
train_daily = (
    train.groupby(["StockCode", "Date"])["Quantity"]
    .sum()
    .reset_index()
)

training_dates = pd.date_range(
    train["Date"].min(),
    train["Date"].max(),
    freq="D"
)

products = policies["StockCode"].astype(str).tolist()
train_daily["StockCode"] = train_daily["StockCode"].astype(str)

train_index = pd.MultiIndex.from_product(
    [products, training_dates],
    names=["StockCode", "Date"]
)

train_complete = (
    train_daily.set_index(["StockCode", "Date"])
    .reindex(train_index, fill_value=0)
    .reset_index()
)

train_stats = (
    train_complete.groupby("StockCode")["Quantity"]
    .agg(["mean", "std"])
    .fillna(0)
)

# Test demand by product; absent products have zero test demand.
test["StockCode"] = test["StockCode"].astype(str)
test_demand = test.groupby("StockCode")["Quantity"].sum()

# Compare demand coverage against each candidate reorder point.
policies["StockCode"] = policies["StockCode"].astype(str)
policies = policies.set_index("StockCode")

policies["Train_Avg_Daily"] = policies.index.map(train_stats["mean"])
policies["Train_Daily_Std"] = policies.index.map(train_stats["std"])
policies[["Train_Avg_Daily", "Train_Daily_Std"]] = (
    policies[["Train_Avg_Daily", "Train_Daily_Std"]].fillna(0)
)

policies["Test_Demand_Units"] = (
    policies.index.map(test_demand).fillna(0)
)

# Demand coverage diagnostic: this is not a full inventory simulation.
# It asks how much test demand is within a simple 30-day stock allowance.
policies["Original_Coverage_Units"] = np.minimum(
    policies["Test_Demand_Units"],
    policies["Reorder_Point"] + 30 * policies["Train_Avg_Daily"]
)

policies["Diagnostic_Coverage_Units"] = np.minimum(
    policies["Test_Demand_Units"],
    policies["Diagnostic_Reorder_Point"]
    + 30 * policies["Train_Avg_Daily"]
)

summary = pd.DataFrame([
    {
        "Policy": "Original safety stock",
        "Products": len(policies),
        "Test_Demand_Units": policies["Test_Demand_Units"].sum(),
        "Coverage_Proxy_Units": policies["Original_Coverage_Units"].sum(),
    },
    {
        "Policy": "Diagnostic safety stock",
        "Products": len(policies),
        "Test_Demand_Units": policies["Test_Demand_Units"].sum(),
        "Coverage_Proxy_Units": policies["Diagnostic_Coverage_Units"].sum(),
    }
])

summary["Coverage_Proxy_Pct"] = (
    summary["Coverage_Proxy_Units"]
    / summary["Test_Demand_Units"].replace(0, np.nan)
    * 100
)

summary_path = output_dir / "inventory_step8_policy_diagnostic.csv"
summary.to_csv(summary_path, index=False)

print("\n--- Policy comparison diagnostic ---")
print(summary.to_string(index=False))
print("\nSaved:", summary_path)
print(
    "\nImportant: this is a coverage proxy, not a simulated replenishment "
    "system. It does not model opening stock, daily orders, supplier receipts, "
    "or inventory carrying costs."
)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

workbook_path = Path("d:/Retail/data/raw/online_retail_ii.xlsx")
policy_path = Path(
    "d:/Retail/reports/inventory/inventory_reorder_points_step7.csv"
)
output_dir = Path("d:/Retail/reports/inventory")
output_dir.mkdir(parents=True, exist_ok=True)

LEAD_TIME_DAYS = 7
REVIEW_DAYS = 1
TRAIN_START = pd.Timestamp("2011-09-11")
TRAIN_END = pd.Timestamp("2011-11-09")
TEST_START = pd.Timestamp("2011-11-10")
TEST_END = pd.Timestamp("2011-12-09")

# Load original transaction data and apply the same eligibility rules.
source = pd.read_excel(
    workbook_path, sheet_name="Year 2010-2011"
)
source["InvoiceDate"] = pd.to_datetime(
    source["InvoiceDate"], errors="coerce"
)
source["Quantity"] = pd.to_numeric(
    source["Quantity"], errors="coerce"
)
source["Price"] = pd.to_numeric(
    source["Price"], errors="coerce"
)

source = source.loc[
    source["StockCode"].notna()
    & source["InvoiceDate"].notna()
    & source["Quantity"].gt(0)
    & source["Price"].gt(0)
].copy()

source["Date"] = source["InvoiceDate"].dt.normalize()
source["StockCode"] = source["StockCode"].astype(str)

source = source.loc[
    source["Date"].between(TRAIN_START, TEST_END)
].copy()

# Daily demand for each product.
daily = source.groupby(
    ["StockCode", "Date"]
)["Quantity"].sum()

policies = pd.read_csv(policy_path)
policies["StockCode"] = policies["StockCode"].astype(str)
policies = policies.set_index("StockCode")

dates = pd.date_range(TEST_START, TEST_END, freq="D")
all_products = policies.index.tolist()

# Use training data only to estimate average daily demand.
train_dates = pd.date_range(TRAIN_START, TRAIN_END, freq="D")
train_index = pd.MultiIndex.from_product(
    [all_products, train_dates],
    names=["StockCode", "Date"]
)

train_daily = (
    daily.reindex(train_index, fill_value=0)
    .rename("Demand")
    .reset_index()
)

train_mean = train_daily.groupby("StockCode")["Demand"].mean()

policies["Train_Avg_Daily"] = (
    policies.index.map(train_mean).astype(float)
)

def simulate_policy(safety_stock_column, policy_name):
    total_demand = 0.0
    fulfilled = 0.0
    unmet = 0.0
    stockout_sku_days = 0
    ending_stock_total = 0.0
    total_orders = 0.0

    for sku in all_products:
        mean_demand = float(policies.at[sku, "Train_Avg_Daily"])
        safety_stock = max(0, float(policies.at[sku, safety_stock_column]))
        reorder_point = max(
            0, mean_demand * LEAD_TIME_DAYS + safety_stock
        )
        order_up_to = reorder_point + mean_demand * 7

        on_hand = 0.0
        pipeline = {}
        sku_demand = 0.0
        sku_fulfilled = 0.0
        sku_unmet = 0.0
        sku_stockout_days = 0
        sku_orders = 0.0

        for day in dates:
            # Receive orders scheduled for today.
            arriving = pipeline.pop(day, 0.0)
            on_hand += arriving

            demand = float(daily.get((sku, day), 0.0))
            sku_demand += demand

            shipped = min(on_hand, demand)
            on_hand -= shipped
            sku_fulfilled += shipped
            sku_unmet += demand - shipped

            if demand > shipped:
                sku_stockout_days += 1

            # Inventory position includes stock and outstanding orders.
            inventory_position = on_hand + sum(pipeline.values())

            if inventory_position <= reorder_point:
                order_qty = max(0.0, order_up_to - inventory_position)
                if order_qty > 0:
                    arrival_day = day + pd.Timedelta(days=LEAD_TIME_DAYS)
                    if arrival_day <= TEST_END:
                        pipeline[arrival_day] = (
                            pipeline.get(arrival_day, 0.0) + order_qty
                        )
                    sku_orders += order_qty

        total_demand += sku_demand
        fulfilled += sku_fulfilled
        unmet += sku_unmet
        stockout_sku_days += sku_stockout_days
        ending_stock_total += on_hand
        total_orders += sku_orders

    return {
        "Policy": policy_name,
        "Products": len(all_products),
        "Test_Demand_Units": total_demand,
        "Fulfilled_Units": fulfilled,
        "Unmet_Demand_Units": unmet,
        "Fill_Rate_Pct": (
            100 * fulfilled / total_demand if total_demand else np.nan
        ),
        "Stockout_SKU_Days": stockout_sku_days,
        "Ending_On_Hand_Units": ending_stock_total,
        "Units_Ordered": total_orders,
    }

# Recalculate both safety-stock approaches using training-period estimates.
policies["Original_Safety_Stock"] = policies["Safety_Stock"]
policies["Diagnostic_Safety_Stock"] = policies[
    "Diagnostic_Safety_Stock"
]

results = pd.DataFrame([
    simulate_policy("Original_Safety_Stock", "Original safety stock"),
    simulate_policy("Diagnostic_Safety_Stock", "Diagnostic safety stock"),
])

results_path = output_dir / "inventory_step9_daily_simulation.csv"
results.to_csv(results_path, index=False)

print("Training period:", TRAIN_START.date(), "to", TRAIN_END.date())
print("Test period:", TEST_START.date(), "to", TEST_END.date())
print("Assumed lead time:", LEAD_TIME_DAYS, "days")
print("\n--- Daily simulation results ---")
print(results.to_string(index=False))
print("\nSaved:", results_path)
print(
    "\nSimulation limitations: zero opening stock, simplified order-up-to "
    "policy, no order constraints, and no real supplier receipt data."
)


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

# Step 10: Validate the Step 9 simulation output

project_root = Path(r"D:\Retail")
simulation_path = (
    project_root
    / "reports"
    / "inventory"
    / "inventory_step9_daily_simulation.csv"
)

results = pd.read_csv(simulation_path)

print("Simulation file loaded successfully.")
print("Shape:", results.shape)
print("\nColumns:")
print(results.columns.tolist())

print("\nSimulation results:")
display(results)

# Basic validation checks
print("\n--- Validation checks ---")

print("Missing values:")
display(results.isnull().sum())

print(
    "Duplicate policy rows:",
    results["Policy"].duplicated().sum()
)

numeric_columns = results.select_dtypes(include=np.number).columns

print("\nNegative values in numeric columns:")
display((results[numeric_columns] < 0).sum())

print("\nFill-rate consistency:")
if {"Test_Demand_Units", "Fulfilled_Units", "Fill_Rate_Pct"}.issubset(results.columns):
    results["Calculated_Fill_Rate_Pct"] = (
        results["Fulfilled_Units"] / results["Test_Demand_Units"] * 100
    )
    display(
        results[
            ["Policy", "Fill_Rate_Pct", "Calculated_Fill_Rate_Pct"]
        ]
    )

print("\nValidation cell completed.")




In [ ]:
from pathlib import Path

project_root = Path(r"D:\Retail")

matches = list(project_root.rglob("run_inventory_backtest.py"))

if matches:
    for file_path in matches:
        print("Found simulation code:")
        print(file_path)
else:
    print("File not found inside D:\\Retail")


In [ ]:
from pathlib import Path

search_locations = [
    Path(r"D:\Retail"),
    Path(r"D:\RetailTorchEnv"),
    Path.cwd()
]

seen = set()

for folder in search_locations:
    if not folder.exists():
        continue

    print(f"\nSearching: {folder}")

    for pattern in ["*inventory*backtest*.py", "*inventory*simulation*.py"]:
        for file_path in folder.rglob(pattern):
            if file_path not in seen:
                seen.add(file_path)
                print(file_path)

if not seen:
    print("\nNo matching Python files found in these locations.")
    print("Your notebook cells and saved CSV outputs may still be available.")


In [ ]:
import numpy as np
import pandas as pd

def simulate_policy_v2(safety_stock_column, policy_name):
    total_demand = 0
    total_fulfilled = 0
    total_unmet = 0
    total_stockout_sku_days = 0
    total_ending_stock = 0
    total_orders_placed = 0
    total_orders_received = 0
    total_outstanding_orders = 0

    test_end = pd.Timestamp(TEST_END)
    simulation_dates = sorted(
        pd.to_datetime(pd.Series(dates)).tolist()
    )

    for sku in all_products:
        mean_demand = max(
            0.0, float(policies.at[sku, "Train_Avg_Daily"])
        )
        safety_stock = max(
            0.0, float(policies.at[sku, safety_stock_column])
        )

        reorder_point = int(np.ceil(
            mean_demand * LEAD_TIME_DAYS + safety_stock
        ))
        order_up_to = int(np.ceil(
            reorder_point + mean_demand * LEAD_TIME_DAYS
        ))

        on_hand = 0
        pipeline = {}

        sku_demand = 0
        sku_fulfilled = 0
        sku_unmet = 0
        sku_stockout_days = 0

        for day in simulation_dates:
            day = pd.Timestamp(day)

            arriving = pipeline.pop(day, 0)
            on_hand += arriving
            total_orders_received += arriving

            raw_demand = float(daily.get((sku, day), 0.0))

            if not np.isfinite(raw_demand) or raw_demand < 0:
                raise ValueError(
                    f"Invalid demand for SKU {sku!r} on {day.date()}"
                )

            demand = int(round(raw_demand))
            sku_demand += demand

            shipped = min(on_hand, demand)
            on_hand -= shipped

            unmet_today = demand - shipped
            sku_fulfilled += shipped
            sku_unmet += unmet_today

            if unmet_today > 0:
                sku_stockout_days += 1

            inventory_position = on_hand + sum(pipeline.values())

            if inventory_position <= reorder_point:
                order_qty = max(
                    0, order_up_to - inventory_position
                )

                if order_qty > 0:
                    arrival_day = (
                        day + pd.Timedelta(days=LEAD_TIME_DAYS)
                    )

                    if arrival_day <= test_end:
                        pipeline[arrival_day] = (
                            pipeline.get(arrival_day, 0) + order_qty
                        )
                        total_orders_placed += order_qty

        total_demand += sku_demand
        total_fulfilled += sku_fulfilled
        total_unmet += sku_unmet
        total_stockout_sku_days += sku_stockout_days
        total_ending_stock += on_hand
        total_outstanding_orders += sum(pipeline.values())

    return {
        "Policy": policy_name,
        "Products": len(all_products),
        "Test_Demand_Units": total_demand,
        "Fulfilled_Units": total_fulfilled,
        "Unmet_Demand_Units": total_unmet,
        "Fill_Rate_Pct": (
            100 * total_fulfilled / total_demand
            if total_demand else 0.0
        ),
        "Stockout_SKU_Days": total_stockout_sku_days,
        "Ending_On_Hand_Units": total_ending_stock,
        "Units_Ordered_Within_Horizon": total_orders_placed,
        "Units_Received_During_Test": total_orders_received,
        "Outstanding_Units_At_End": total_outstanding_orders,
    }


results_step11 = pd.DataFrame([
    simulate_policy_v2(
        "Safety_Stock", "Original safety stock"
    ),
    simulate_policy_v2(
        "Diagnostic_Safety_Stock", "Diagnostic safety stock"
    ),
])

results_step11["Calculated_Fill_Rate_Pct"] = (
    100 * results_step11["Fulfilled_Units"]
    / results_step11["Test_Demand_Units"].replace(0, np.nan)
)

assert results_step11["Policy"].is_unique
assert results_step11.isna().sum().sum() == 0
assert (
    results_step11["Fulfilled_Units"]
    + results_step11["Unmet_Demand_Units"]
    == results_step11["Test_Demand_Units"]
).all()
assert (
    results_step11.select_dtypes(include=np.number) >= 0
).all().all()

output_dir.mkdir(parents=True, exist_ok=True)

results_path_step11 = (
    output_dir / "inventory_step11_validated_simulation.csv"
)
results_step11.to_csv(results_path_step11, index=False)

print("Training period:", TRAIN_START.date(), "to", TRAIN_END.date())
print("Test period:", TEST_START.date(), "to", TEST_END.date())
print("Assumed lead time:", LEAD_TIME_DAYS, "days")
print("\n--- Step 11 simulation results ---")
print(results_step11.to_string(index=False))
print("\nSaved:", results_path_step11)


In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"D:\Retail")

baseline_path = BASE_DIR / "reports" / "inventory" / "product_demand_baseline_step4.csv"
reorder_path = BASE_DIR / "reports" / "inventory" / "inventory_reorder_points_step5.csv"
diagnostic_path = BASE_DIR / "reports" / "inventory" / "inventory_reorder_points_step7.csv"

for path in [baseline_path, reorder_path, diagnostic_path]:
    print(f"\nFile: {path.name}")
    if path.exists():
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(3).to_string(index=False))
    else:
        print("FILE NOT FOUND")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(r"D:\Retail")
RAW_FILE = BASE_DIR / "data" / "raw" / "online_retail_ii.xlsx"

TRAIN_START = pd.Timestamp("2011-09-11")
TRAIN_END = pd.Timestamp("2011-11-09")
TEST_START = pd.Timestamp("2011-11-10")
TEST_END = pd.Timestamp("2011-12-09")

print("STEP 12: TRAINING-DATA LEAKAGE CHECK")
print("-" * 55)

# Load the original workbook, preserving original StockCode values.
sheets = pd.read_excel(
    RAW_FILE,
    sheet_name=["Year 2009-2010", "Year 2010-2011"],
    usecols=[
        "Invoice", "StockCode", "Quantity",
        "InvoiceDate", "Price"
    ]
)

df = pd.concat(sheets.values(), ignore_index=True)

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)
df["StockCode"] = df["StockCode"].astype("string").str.strip()

df = df[
    df["StockCode"].notna()
    & df["InvoiceDate"].notna()
    & (df["Quantity"] > 0)
    & (df["Price"] > 0)
].copy()

# Training-period demand only.
train = df[
    df["InvoiceDate"].between(
        TRAIN_START, TRAIN_END + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    )
].copy()

# Test-period demand, kept separate.
test = df[
    df["InvoiceDate"].between(
        TEST_START, TEST_END + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    )
].copy()

calendar = pd.date_range(TRAIN_START, TRAIN_END, freq="D")

daily_train = (
    train.groupby(["StockCode", train["InvoiceDate"].dt.normalize()])["Quantity"]
    .sum()
    .rename("Demand")
    .reset_index()
    .rename(columns={"InvoiceDate": "Date"})
)

# Include zero-demand days for every SKU in the training set.
products = sorted(train["StockCode"].unique())
full_index = pd.MultiIndex.from_product(
    [products, calendar], names=["StockCode", "Date"]
)

daily_full = (
    daily_train.set_index(["StockCode", "Date"])["Demand"]
    .reindex(full_index, fill_value=0)
    .rename("Demand")
    .reset_index()
)

training_stats = (
    daily_full.groupby("StockCode")["Demand"]
    .agg(
        Training_Total_Units="sum",
        Training_Avg_Daily_Demand="mean",
        Training_Daily_Demand_Std="std"
    )
    .reset_index()
)

training_stats["Training_Daily_Demand_Std"] = (
    training_stats["Training_Daily_Demand_Std"].fillna(0)
)

# Compare against the existing full-window baseline.
baseline = pd.read_csv(
    BASE_DIR / "reports" / "inventory" / "product_demand_baseline_step4.csv",
    dtype={"StockCode": "string"}
)
baseline["StockCode"] = baseline["StockCode"].str.strip()

comparison = training_stats.merge(
    baseline[["StockCode", "Avg_Daily_Demand", "Daily_Demand_Std"]],
    on="StockCode",
    how="inner"
)

comparison["Mean_Differs"] = ~np.isclose(
    comparison["Training_Avg_Daily_Demand"],
    comparison["Avg_Daily_Demand"],
    rtol=1e-4,
    atol=1e-4
)

comparison["Std_Differs"] = ~np.isclose(
    comparison["Training_Daily_Demand_Std"],
    comparison["Daily_Demand_Std"],
    rtol=1e-4,
    atol=1e-4
)

print("Training dates:", TRAIN_START.date(), "to", TRAIN_END.date())
print("Test dates:", TEST_START.date(), "to", TEST_END.date())
print("Training rows:", len(train))
print("Test rows:", len(test))
print("Training SKUs:", train["StockCode"].nunique())
print("Test SKUs:", test["StockCode"].nunique())
print("SKUs compared with baseline:", len(comparison))
print("SKUs with different mean demand:", int(comparison["Mean_Differs"].sum()))
print("SKUs with different demand standard deviation:", int(comparison["Std_Differs"].sum()))

print("\nLargest differences between training-only and full-window means:")
print(
    comparison.assign(
        Absolute_Mean_Difference=(
            comparison["Training_Avg_Daily_Demand"]
            - comparison["Avg_Daily_Demand"]
        ).abs()
    )
    .sort_values("Absolute_Mean_Difference", ascending=False)
    .head(10)[[
        "StockCode",
        "Training_Avg_Daily_Demand",
        "Avg_Daily_Demand",
        "Training_Daily_Demand_Std",
        "Daily_Demand_Std",
        "Mean_Differs",
        "Std_Differs"
    ]]
    .to_string(index=False)
)

print("\nNo source files were modified.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(r"D:\Retail")
RAW_FILE = BASE_DIR / "data" / "raw" / "online_retail_ii.xlsx"
OUTPUT_DIR = BASE_DIR / "reports" / "inventory"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_START = pd.Timestamp("2011-09-11")
TRAIN_END = pd.Timestamp("2011-11-09")
LEAD_TIME_DAYS = 7
SERVICE_Z = 1.2816

# Load original SKU codes from the workbook.
sheets = pd.read_excel(
    RAW_FILE,
    sheet_name=["Year 2009-2010", "Year 2010-2011"],
    usecols=["StockCode", "Quantity", "InvoiceDate", "Price"]
)
df = pd.concat(sheets.values(), ignore_index=True)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["StockCode"] = df["StockCode"].astype("string").str.strip()

df = df[
    df["StockCode"].notna()
    & df["InvoiceDate"].notna()
    & (df["Quantity"] > 0)
    & (df["Price"] > 0)
].copy()

train = df[
    df["InvoiceDate"].between(
        TRAIN_START,
        TRAIN_END + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    )
].copy()

calendar = pd.date_range(TRAIN_START, TRAIN_END, freq="D")
products = sorted(train["StockCode"].unique())

daily = (
    train.assign(Date=train["InvoiceDate"].dt.normalize())
    .groupby(["StockCode", "Date"])["Quantity"]
    .sum()
)

full_index = pd.MultiIndex.from_product(
    [products, calendar], names=["StockCode", "Date"]
)

daily_full = (
    daily.reindex(full_index, fill_value=0)
    .rename("Daily_Demand")
    .reset_index()
)

stats = (
    daily_full.groupby("StockCode")["Daily_Demand"]
    .agg(
        Training_Total_Units="sum",
        Avg_Daily_Demand="mean",
        Daily_Demand_Std="std",
        Active_Days=lambda x: int((x > 0).sum()),
        Max_Daily_Demand="max"
    )
    .reset_index()
)

stats["Daily_Demand_Std"] = stats["Daily_Demand_Std"].fillna(0)

stats["Lead_Time_Demand"] = (
    stats["Avg_Daily_Demand"] * LEAD_TIME_DAYS
)

stats["Safety_Stock"] = np.ceil(
    SERVICE_Z
    * stats["Daily_Demand_Std"]
    * np.sqrt(LEAD_TIME_DAYS)
).astype(int)

stats["Reorder_Point"] = np.ceil(
    stats["Lead_Time_Demand"] + stats["Safety_Stock"]
).astype(int)

stats["Suggested_Order_Up_To"] = np.ceil(
    stats["Reorder_Point"]
    + stats["Avg_Daily_Demand"] * LEAD_TIME_DAYS
).astype(int)

stats["Demand_Review_Flag"] = np.where(
    (stats["Active_Days"] < 5)
    | (
        stats["Daily_Demand_Std"]
        > 2 * stats["Avg_Daily_Demand"].clip(lower=1)
    ),
    "Review demand pattern",
    "Standard review"
)

# Diagnostic alternative for sensitivity analysis only.
stats["Diagnostic_Demand_Std"] = np.sqrt(
    stats["Avg_Daily_Demand"].clip(lower=0)
)

stats["Diagnostic_Safety_Stock"] = np.ceil(
    SERVICE_Z
    * stats["Diagnostic_Demand_Std"]
    * np.sqrt(LEAD_TIME_DAYS)
).astype(int)

stats["Diagnostic_Reorder_Point"] = np.ceil(
    stats["Lead_Time_Demand"] + stats["Diagnostic_Safety_Stock"]
).astype(int)

output_path = OUTPUT_DIR / "inventory_training_only_policies_step13.csv"
stats.to_csv(output_path, index=False)

assert stats["StockCode"].is_unique
assert stats["Reorder_Point"].ge(0).all()
assert stats["Safety_Stock"].ge(0).all()
assert stats["Avg_Daily_Demand"].ge(0).all()
assert stats.notna().all().all()

print("STEP 13: TRAINING-ONLY POLICIES CREATED")
print("Training period:", TRAIN_START.date(), "to", TRAIN_END.date())
print("Training rows:", len(train))
print("Products:", len(stats))
print("Missing values:", int(stats.isna().sum().sum()))
print("Duplicate StockCodes:", int(stats["StockCode"].duplicated().sum()))
print("Saved:", output_path)
print("\nTop 10 products by training demand:")
print(
    stats.sort_values("Training_Total_Units", ascending=False)
    [[
        "StockCode", "Training_Total_Units", "Avg_Daily_Demand",
        "Daily_Demand_Std", "Safety_Stock", "Reorder_Point",
        "Suggested_Order_Up_To", "Demand_Review_Flag"
    ]]
    .head(10)
    .to_string(index=False)
)


In [ ]:
from pathlib import Path
from collections import defaultdict, deque
import pandas as pd
import numpy as np

BASE_DIR = Path(r"D:\Retail")
RAW_FILE = BASE_DIR / "data" / "raw" / "online_retail_ii.xlsx"
POLICY_FILE = BASE_DIR / "reports" / "inventory" / "inventory_training_only_policies_step13.csv"
OUTPUT_FILE = BASE_DIR / "reports" / "inventory" / "inventory_step14_training_only_simulation.csv"

TRAIN_START = pd.Timestamp("2011-09-11")
TRAIN_END = pd.Timestamp("2011-11-09")
TEST_START = pd.Timestamp("2011-11-10")
TEST_END = pd.Timestamp("2011-12-09")
LEAD_TIME_DAYS = 7

# Load the original workbook and preserve original SKU codes.
sheets = pd.read_excel(
    RAW_FILE,
    sheet_name=["Year 2009-2010", "Year 2010-2011"],
    usecols=["StockCode", "Quantity", "InvoiceDate", "Price"]
)
df = pd.concat(sheets.values(), ignore_index=True)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["StockCode"] = df["StockCode"].astype("string").str.strip()

df = df[
    df["StockCode"].notna()
    & df["InvoiceDate"].notna()
    & (df["Quantity"] > 0)
    & (df["Price"] > 0)
].copy()

df["Date"] = df["InvoiceDate"].dt.normalize()

test = df[
    df["Date"].between(TEST_START, TEST_END)
].copy()

daily_test = (
    test.groupby(["StockCode", "Date"])["Quantity"]
    .sum()
    .unstack(fill_value=0)
)

calendar = pd.date_range(TEST_START, TEST_END, freq="D")
daily_test = daily_test.reindex(columns=calendar, fill_value=0)

policies = pd.read_csv(
    POLICY_FILE,
    dtype={"StockCode": "string"}
)
policies["StockCode"] = policies["StockCode"].str.strip()
policies = policies.set_index("StockCode")

# Include all training SKUs. SKUs first appearing in test are reported separately.
test_only_skus = sorted(set(daily_test.index) - set(policies.index))
daily_test = daily_test.reindex(
    index=policies.index,
    columns=calendar,
    fill_value=0
)

def simulate_policy(policy_name, safety_col, reorder_col, order_up_to_col):
    total_demand = 0
    total_fulfilled = 0
    total_unmet = 0
    total_stockout_sku_days = 0
    total_ending_stock = 0
    total_ordered = 0
    total_received = 0

    for sku, policy in policies.iterrows():
        mean_daily = float(policy["Avg_Daily_Demand"])
        reorder_point = int(policy[reorder_col])
        order_up_to = int(policy[order_up_to_col])

        safety_stock = int(policy[safety_col])
        on_hand = 0
        pipeline = defaultdict(int)
        order_queue = deque()

        sku_demand = 0
        sku_fulfilled = 0
        sku_unmet = 0
        sku_stockout_days = 0
        sku_ordered = 0
        sku_received = 0

        for day in calendar:
            # Receive orders due today.
            on_hand += pipeline.pop(day, 0)
            sku_received += pipeline.get(day, 0) if False else 0

            # Demand is lost if unavailable; it is not backordered.
            demand = int(round(float(daily_test.at[sku, day])))
            fulfilled = min(on_hand, demand)
            unmet = demand - fulfilled
            on_hand -= fulfilled

            sku_demand += demand
            sku_fulfilled += fulfilled
            sku_unmet += unmet

            if unmet > 0:
                sku_stockout_days += 1

            # Inventory position = on-hand plus outstanding orders.
            inventory_position = on_hand + sum(pipeline.values())

            if inventory_position <= reorder_point:
                order_qty = max(0, order_up_to - inventory_position)
                arrival_day = day + pd.Timedelta(days=LEAD_TIME_DAYS)

                # Count only orders that can arrive inside the test horizon.
                if order_qty > 0 and arrival_day <= TEST_END:
                    pipeline[arrival_day] = pipeline.get(arrival_day, 0) + order_qty
                    sku_ordered += order_qty

        # All orders were placed only if they could arrive within the horizon.
        sku_received = sku_ordered

        total_demand += sku_demand
        total_fulfilled += sku_fulfilled
        total_unmet += sku_unmet
        total_stockout_sku_days += sku_stockout_days
        total_ending_stock += on_hand
        total_ordered += sku_ordered
        total_received += sku_received

    fill_rate = (
        100 * total_fulfilled / total_demand
        if total_demand else 0
    )

    return {
        "Policy": policy_name,
        "Products": len(policies),
        "Test_Demand_Units": total_demand,
        "Fulfilled_Units": total_fulfilled,
        "Unmet_Demand_Units": total_unmet,
        "Fill_Rate_Pct": round(fill_rate, 4),
        "Stockout_SKU_Days": total_stockout_sku_days,
        "Ending_On_Hand_Units": total_ending_stock,
        "Units_Ordered_Within_Horizon": total_ordered,
        "Units_Received_During_Test": total_received,
    }

results = pd.DataFrame([
    simulate_policy(
        "Training-only original safety stock",
        "Safety_Stock",
        "Reorder_Point",
        "Suggested_Order_Up_To"
    ),
    simulate_policy(
        "Training-only diagnostic safety stock",
        "Diagnostic_Safety_Stock",
        "Diagnostic_Reorder_Point",
        "Suggested_Order_Up_To"
    )
])

results["Calculated_Fill_Rate_Pct"] = (
    100 * results["Fulfilled_Units"] / results["Test_Demand_Units"]
).round(4)

results["Demand_Reconciliation_Difference"] = (
    results["Test_Demand_Units"]
    - results["Fulfilled_Units"]
    - results["Unmet_Demand_Units"]
)

assert (results["Demand_Reconciliation_Difference"] == 0).all()
assert np.allclose(
    results["Fill_Rate_Pct"],
    results["Calculated_Fill_Rate_Pct"]
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_FILE, index=False)

print("STEP 14: TRAINING-ONLY INVENTORY SIMULATION")
print("Training period:", TRAIN_START.date(), "to", TRAIN_END.date())
print("Test period:", TEST_START.date(), "to", TEST_END.date())
print("Assumed lead time:", LEAD_TIME_DAYS, "days")
print("Training SKUs:", len(policies))
print("SKUs appearing only in test:", len(test_only_skus))
print("\nResults:")
print(results.to_string(index=False))
print("\nSaved:", OUTPUT_FILE)
print("\nNote: Simulation starts with zero inventory and assumes lost sales.")

In [ ]:
from pathlib import Path
from collections import defaultdict, deque
import pandas as pd
import numpy as np

BASE_DIR = Path(r"D:\Retail")
RAW_FILE = BASE_DIR / "data" / "raw" / "online_retail_ii.xlsx"
POLICY_FILE = BASE_DIR / "reports" / "inventory" / "inventory_training_only_policies_step13.csv"
OUTPUT_FILE = BASE_DIR / "reports" / "inventory" / "inventory_step14_training_only_simulation.csv"

TRAIN_START = pd.Timestamp("2011-09-11")
TRAIN_END = pd.Timestamp("2011-11-09")
TEST_START = pd.Timestamp("2011-11-10")
TEST_END = pd.Timestamp("2011-12-09")
LEAD_TIME_DAYS = 7

# Load the original workbook and preserve original SKU codes.
sheets = pd.read_excel(
    RAW_FILE,
    sheet_name=["Year 2009-2010", "Year 2010-2011"],
    usecols=["StockCode", "Quantity", "InvoiceDate", "Price"]
)
df = pd.concat(sheets.values(), ignore_index=True)

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["StockCode"] = df["StockCode"].astype("string").str.strip()

df = df[
    df["StockCode"].notna()
    & df["InvoiceDate"].notna()
    & (df["Quantity"] > 0)
    & (df["Price"] > 0)
].copy()

df["Date"] = df["InvoiceDate"].dt.normalize()

test = df[
    df["Date"].between(TEST_START, TEST_END)
].copy()

daily_test = (
    test.groupby(["StockCode", "Date"])["Quantity"]
    .sum()
    .unstack(fill_value=0)
)

calendar = pd.date_range(TEST_START, TEST_END, freq="D")
daily_test = daily_test.reindex(columns=calendar, fill_value=0)

policies = pd.read_csv(
    POLICY_FILE,
    dtype={"StockCode": "string"}
)
policies["StockCode"] = policies["StockCode"].str.strip()
policies = policies.set_index("StockCode")

# Include all training SKUs. SKUs first appearing in test are reported separately.
test_only_skus = sorted(set(daily_test.index) - set(policies.index))
daily_test = daily_test.reindex(
    index=policies.index,
    columns=calendar,
    fill_value=0
)

def simulate_policy(policy_name, safety_col, reorder_col, order_up_to_col):
    total_demand = 0
    total_fulfilled = 0
    total_unmet = 0
    total_stockout_sku_days = 0
    total_ending_stock = 0
    total_ordered = 0
    total_received = 0

    for sku, policy in policies.iterrows():
        mean_daily = float(policy["Avg_Daily_Demand"])
        reorder_point = int(policy[reorder_col])
        order_up_to = int(policy[order_up_to_col])

        safety_stock = int(policy[safety_col])
        on_hand = 0
        pipeline = defaultdict(int)
        order_queue = deque()

        sku_demand = 0
        sku_fulfilled = 0
        sku_unmet = 0
        sku_stockout_days = 0
        sku_ordered = 0
        sku_received = 0

        for day in calendar:
            # Receive orders due today.
            on_hand += pipeline.pop(day, 0)
            sku_received += pipeline.get(day, 0) if False else 0

            # Demand is lost if unavailable; it is not backordered.
            demand = int(round(float(daily_test.at[sku, day])))
            fulfilled = min(on_hand, demand)
            unmet = demand - fulfilled
            on_hand -= fulfilled

            sku_demand += demand
            sku_fulfilled += fulfilled
            sku_unmet += unmet

            if unmet > 0:
                sku_stockout_days += 1

            # Inventory position = on-hand plus outstanding orders.
            inventory_position = on_hand + sum(pipeline.values())

            if inventory_position <= reorder_point:
                order_qty = max(0, order_up_to - inventory_position)
                arrival_day = day + pd.Timedelta(days=LEAD_TIME_DAYS)

                # Count only orders that can arrive inside the test horizon.
                if order_qty > 0 and arrival_day <= TEST_END:
                    pipeline[arrival_day] = pipeline.get(arrival_day, 0) + order_qty
                    sku_ordered += order_qty

        # All orders were placed only if they could arrive within the horizon.
        sku_received = sku_ordered

        total_demand += sku_demand
        total_fulfilled += sku_fulfilled
        total_unmet += sku_unmet
        total_stockout_sku_days += sku_stockout_days
        total_ending_stock += on_hand
        total_ordered += sku_ordered
        total_received += sku_received

    fill_rate = (
        100 * total_fulfilled / total_demand
        if total_demand else 0
    )

    return {
        "Policy": policy_name,
        "Products": len(policies),
        "Test_Demand_Units": total_demand,
        "Fulfilled_Units": total_fulfilled,
        "Unmet_Demand_Units": total_unmet,
        "Fill_Rate_Pct": round(fill_rate, 4),
        "Stockout_SKU_Days": total_stockout_sku_days,
        "Ending_On_Hand_Units": total_ending_stock,
        "Units_Ordered_Within_Horizon": total_ordered,
        "Units_Received_During_Test": total_received,
    }

results = pd.DataFrame([
    simulate_policy(
        "Training-only original safety stock",
        "Safety_Stock",
        "Reorder_Point",
        "Suggested_Order_Up_To"
    ),
    simulate_policy(
        "Training-only diagnostic safety stock",
        "Diagnostic_Safety_Stock",
        "Diagnostic_Reorder_Point",
        "Suggested_Order_Up_To"
    )
])

results["Calculated_Fill_Rate_Pct"] = (
    100 * results["Fulfilled_Units"] / results["Test_Demand_Units"]
).round(4)

results["Demand_Reconciliation_Difference"] = (
    results["Test_Demand_Units"]
    - results["Fulfilled_Units"]
    - results["Unmet_Demand_Units"]
)

assert (results["Demand_Reconciliation_Difference"] == 0).all()
assert np.allclose(
    results["Fill_Rate_Pct"],
    results["Calculated_Fill_Rate_Pct"]
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_FILE, index=False)

print("STEP 14: TRAINING-ONLY INVENTORY SIMULATION")
print("Training period:", TRAIN_START.date(), "to", TRAIN_END.date())
print("Test period:", TEST_START.date(), "to", TEST_END.date())
print("Assumed lead time:", LEAD_TIME_DAYS, "days")
print("Training SKUs:", len(policies))
print("SKUs appearing only in test:", len(test_only_skus))
print("\nResults:")
print(results.to_string(index=False))
print("\nSaved:", OUTPUT_FILE)
print("\nNote: Simulation starts with zero inventory and assumes lost sales.")

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"D:\Retail")
RAW_FILE = BASE_DIR / "data" / "raw" / "online_retail_ii.xlsx"
POLICY_FILE = (
    BASE_DIR / "reports" / "inventory"
    / "inventory_training_only_policies_step13.csv"
)

TEST_START = pd.Timestamp("2011-11-10")
TEST_END = pd.Timestamp("2011-12-09")

sheets = pd.read_excel(
    RAW_FILE,
    sheet_name=["Year 2009-2010", "Year 2010-2011"],
    usecols=["StockCode", "Quantity", "InvoiceDate", "Price"]
)

df = pd.concat(sheets.values(), ignore_index=True)

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)
df["StockCode"] = (
    df["StockCode"].astype("string").str.strip()
)

df = df[
    df["StockCode"].notna()
    & df["InvoiceDate"].notna()
    & (df["Quantity"] > 0)
    & (df["Price"] > 0)
].copy()

df["Date"] = df["InvoiceDate"].dt.normalize()

test = df[
    df["Date"].between(TEST_START, TEST_END)
].copy()

policies = pd.read_csv(
    POLICY_FILE,
    dtype={"StockCode": "string"}
)

training_skus = set(policies["StockCode"].str.strip())
test_skus = set(test["StockCode"])

new_skus = test[
    ~test["StockCode"].isin(training_skus)
]

total_test_units = int(test["Quantity"].sum())

covered_units = int(
    test.loc[
        test["StockCode"].isin(training_skus),
        "Quantity"
    ].sum()
)

new_sku_units = int(new_skus["Quantity"].sum())

print("STEP 15: TEST-DEMAND COVERAGE VALIDATION")
print("Test period:", TEST_START.date(), "to", TEST_END.date())
print("Total test demand:", total_test_units)
print("Demand from training SKUs:", covered_units)
print("Demand from new test-only SKUs:", new_sku_units)
print("Training SKUs:", len(training_skus))
print("Test SKUs:", len(test_skus))
print("Test-only SKUs:", new_skus["StockCode"].nunique())
print(
    "Demand coverage (%):",
    round(100 * covered_units / total_test_units, 2)
)
print(
    "New-SKU demand share (%):",
    round(100 * new_sku_units / total_test_units, 2)
)

print("\nTop 15 test-only SKUs by units:")
print(
    new_skus.groupby("StockCode")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .to_string()
)

assert covered_units + new_sku_units == total_test_units

print("\nValidation passed: test demand is fully reconciled.")
